# Task: 记忆更新

## Objective / 目标
- 实现记忆更新的核心操作：覆盖、合并、版本化
- 实现冲突消解机制
- 实现遗忘策略与长期稳定性维护

## Scope & Constraints / 范围与约束
- 中文：一个 Notebook 只做一个任务闭环（输入 → 方法 → 输出/验证）。
- English: One notebook = one task loop (input → method → output/validation).
- 核心逻辑应可迁移为 `.py`：尽量纯函数、无隐式全局状态，I/O 放在边缘。

In [1]:
# ============================================================
# 基础依赖与数据结构定义
# ============================================================
import os
import uuid
import json
import math
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict
from enum import Enum
from collections import defaultdict
import copy

# OpenAI API 依赖
from openai import OpenAI

# 从环境变量或直接配置加载API密钥
# 方式1: 使用dotenv加载.env文件
# from dotenv import load_dotenv
# load_dotenv()

# 方式2: 直接配置（请替换为你的实际配置）
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
MODEL_NAME = "gpt-4.1-mini"  # 使用gpt-4.1-mini模型

# 初始化OpenAI客户端
client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL
)

print(f"OpenAI客户端已初始化")
print(f"  Base URL: {OPENAI_BASE_URL}")
print(f"  Model: {MODEL_NAME}")


OpenAI客户端已初始化
  Base URL: https://api.openai.com/v1
  Model: gpt-4.1-mini


In [2]:
# ============================================================
# 枚举类型定义
# ============================================================

class MemoryType(Enum):
    """记忆类型枚举"""
    USER_MEMORY = "UserMemory"
    SYSTEM_MEMORY = "SystemMemory"
    FACT = "fact"
    EVENT = "event"
    PREFERENCE = "preference"


class MemoryStatus(Enum):
    """记忆状态枚举"""
    ACTIVATED = "activated"      # 活跃状态，可被检索
    ARCHIVED = "archived"        # 归档状态，移至冷存储
    DEPRECATED = "deprecated"    # 废弃状态，标记为过时
    DELETED = "deleted"          # 删除状态


class UpdateAction(Enum):
    """更新操作类型"""
    OVERWRITE = "overwrite"      # 覆盖：新信息完全替代旧信息
    MERGE = "merge"              # 合并：新旧信息融合
    VERSION = "version"          # 版本化：保留历史，创建新版本
    CONFLICT = "conflict"        # 冲突：需要消解


class ConflictType(Enum):
    """冲突类型"""
    FACTUAL = "factual"          # 事实冲突：硬性矛盾
    PREFERENCE = "preference"    # 偏好冲突：行为偏离
    TEMPORAL = "temporal"        # 时间冲突：时效性问题


class SourceCredibility(Enum):
    """来源可信度等级"""
    USER_EXPLICIT = 1.0          # 用户明确陈述
    SYSTEM_INFERENCE = 0.7       # 系统推断
    OFFICIAL_DOC = 0.9           # 官方文档
    THIRD_PARTY = 0.5            # 第三方来源


In [3]:
# ============================================================
# 核心数据结构
# ============================================================

@dataclass
class MemoryNode:
    """
    记忆节点：系统的基本存储单元
    
    Attributes:
        id: 唯一标识符
        key: 记忆关键词/标题
        memory: 记忆内容摘要
        tags: 标签列表
        type: 记忆类型
        confidence: 置信度 (0-1)
        created_at: 创建时间
        updated_at: 更新时间
        user_id: 用户ID
        session_id: 会话ID
        status: 记忆状态
        source_type: 来源类型
        source_credibility: 来源可信度
        access_count: 访问次数（用于热度计算）
        last_accessed: 最后访问时间
        decay_score: 衰减分数（用于遗忘机制）
        version: 版本号
        parent_version_id: 父版本ID（用于版本链）
        embedding: 向量表示（可选）
    """
    id: str
    key: str
    memory: str
    tags: List[str]
    type: str
    confidence: float
    created_at: str
    updated_at: str
    user_id: str
    session_id: str
    status: str = MemoryStatus.ACTIVATED.value
    source_type: str = "user_explicit"
    source_credibility: float = 1.0
    access_count: int = 0
    last_accessed: Optional[str] = None
    decay_score: float = 1.0
    version: int = 1
    parent_version_id: Optional[str] = None
    embedding: Optional[List[float]] = None
    
    def to_dict(self) -> Dict:
        """转换为字典"""
        return asdict(self)
    
    @classmethod
    def from_dict(cls, data: Dict) -> 'MemoryNode':
        """从字典创建"""
        return cls(**{k: v for k, v in data.items() if k in cls.__dataclass_fields__})
    
    def get_timestamp(self) -> datetime:
        """获取创建时间的datetime对象"""
        return datetime.fromisoformat(self.created_at.replace('Z', '+00:00'))
    
    def get_effective_score(self) -> float:
        """计算综合有效分数：置信度 × 衰减分数 × 来源可信度"""
        return self.confidence * self.decay_score * self.source_credibility


@dataclass
class VersionedMemory:
    """
    版本化记忆：记录记忆的完整版本历史
    
    Attributes:
        memory_id: 当前记忆ID
        version_chain: 版本链（从旧到新的ID列表）
        current_version: 当前版本号
        change_log: 变更日志
    """
    memory_id: str
    version_chain: List[str] = field(default_factory=list)
    current_version: int = 1
    change_log: List[Dict] = field(default_factory=list)


@dataclass
class ConflictRecord:
    """
    冲突记录：记录检测到的记忆冲突
    
    Attributes:
        conflict_id: 冲突ID
        memory_id_a: 记忆A的ID
        memory_id_b: 记忆B的ID
        conflict_type: 冲突类型
        description: 冲突描述
        resolution: 解决方案
        resolved: 是否已解决
        created_at: 创建时间
    """
    conflict_id: str
    memory_id_a: str
    memory_id_b: str
    conflict_type: ConflictType
    description: str
    resolution: Optional[str] = None
    resolved: bool = False
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class MemoryUpdateConfig:
    """记忆更新配置"""
    # 衰减参数
    decay_rate: float = 0.1              # 基础衰减率
    decay_interval_hours: int = 24       # 衰减计算间隔
    min_decay_score: float = 0.1         # 最小衰减分数
    
    # 遗忘阈值
    archive_threshold: float = 0.3       # 归档阈值
    delete_threshold: float = 0.1        # 删除阈值
    
    # 合并参数
    similarity_threshold: float = 0.7    # 相似度阈值（用于判断是否合并）
    merge_confidence_boost: float = 0.1  # 合并后置信度提升
    
    # 冲突消解参数
    time_priority_weight: float = 0.4    # 时效性权重
    source_priority_weight: float = 0.4  # 来源可信度权重
    confidence_weight: float = 0.2       # 置信度权重
    
    # 巩固参数
    consolidation_access_threshold: int = 5  # 触发巩固的访问次数
    
    verbose: bool = True


In [4]:
# ============================================================
# 检索模块与LLM服务模块（使用OpenAI API）
# ============================================================

class SimpleRetriever:
    """
    简单检索器：基于关键词匹配
    实际生产环境应替换为向量检索（如使用embedding + 向量数据库）
    """
    
    def __init__(self, memory_store: Dict[str, MemoryNode]):
        self.memory_store = memory_store
    
    def retrieve(self, query: str, top_k: int = 5, 
                 user_id: Optional[str] = None) -> List[Tuple[MemoryNode, float]]:
        """检索与query相关的记忆"""
        results = []
        query_words = set(query.lower().split())
        
        for memory in self.memory_store.values():
            if memory.status != MemoryStatus.ACTIVATED.value:
                continue
            if user_id and memory.user_id != user_id:
                continue
            
            memory_words = set(memory.memory.lower().split())
            memory_words.update(memory.key.lower().split())
            memory_words.update(tag.lower() for tag in memory.tags)
            
            overlap = len(query_words & memory_words)
            if overlap > 0:
                similarity = overlap / max(len(query_words), 1)
                results.append((memory, similarity))
        
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]
    
    def find_similar(self, memory: MemoryNode, 
                     threshold: float = 0.5) -> List[Tuple[MemoryNode, float]]:
        """查找与给定记忆相似的其他记忆"""
        query = f"{memory.key} {memory.memory} {' '.join(memory.tags)}"
        results = self.retrieve(query, top_k=10, user_id=memory.user_id)
        return [(m, s) for m, s in results if m.id != memory.id and s >= threshold]


class LLMService:
    """
    LLM服务：使用OpenAI API进行记忆相关的智能处理
    包含冲突检测、记忆合并、冲突消解等功能
    """
    
    def __init__(self, model: str = None):
        self.model = model or MODEL_NAME
        self.client = client
    
    def _call_llm(self, system_prompt: str, user_prompt: str, 
                  temperature: float = 0.3) -> str:
        """调用LLM的通用方法"""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"[LLMService] API调用失败: {e}")
            return ""
    
    def _parse_json_response(self, response: str) -> Optional[Dict]:
        """解析LLM的JSON响应"""
        try:
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0]
            elif "```" in response:
                response = response.split("```")[1].split("```")[0]
            return json.loads(response.strip())
        except json.JSONDecodeError as e:
            print(f"[LLMService] JSON解析失败: {e}")
            return None
    
    def detect_conflict(self, memory_a: MemoryNode, memory_b: MemoryNode) -> Optional[Dict]:
        """
        使用LLM检测两条记忆是否存在冲突
        """
        system_prompt = """你是一个记忆冲突检测专家。判断两条记忆是否存在逻辑冲突。

冲突类型：
1. FACTUAL（事实冲突）：事实层面互相矛盾
2. PREFERENCE（偏好冲突）：长期偏好与短期行为偏离
3. TEMPORAL（时间冲突）：时间相关信息矛盾

输出JSON格式：
- 有冲突：{"has_conflict": true, "conflict_type": "FACTUAL/PREFERENCE/TEMPORAL", "description": "冲突描述", "severity": "high/medium/low"}
- 无冲突：{"has_conflict": false}

只输出JSON。"""

        user_prompt = f"""判断以下两条记忆是否冲突：

记忆A：{memory_a.memory}（时间：{memory_a.created_at}，来源：{memory_a.source_type}）

记忆B：{memory_b.memory}（时间：{memory_b.created_at}，来源：{memory_b.source_type}）"""

        response = self._call_llm(system_prompt, user_prompt)
        result = self._parse_json_response(response)
        
        if result and result.get("has_conflict"):
            conflict_type_map = {
                "FACTUAL": ConflictType.FACTUAL,
                "PREFERENCE": ConflictType.PREFERENCE,
                "TEMPORAL": ConflictType.TEMPORAL
            }
            return {
                "has_conflict": True,
                "conflict_type": conflict_type_map.get(result.get("conflict_type", "FACTUAL"), ConflictType.FACTUAL),
                "description": result.get("description", "检测到冲突"),
                "severity": result.get("severity", "medium")
            }
        return None
    
    def merge_memories(self, memories: List[MemoryNode]) -> str:
        """
        使用LLM合并多条记忆为一条综合描述
        """
        if not memories:
            return ""
        if len(memories) == 1:
            return memories[0].memory
        
        system_prompt = """你是一个记忆整合专家。将多条相关记忆合并为一条简洁完整的综合记忆。

原则：
1. 保留所有重要信息，去除冗余
2. 按时间顺序整理（如有变化）
3. 以最新信息为准
4. 保持客观简洁

直接输出合并后的记忆内容，不要其他解释。"""

        memory_list = [f"{i}. [{m.created_at}] {m.memory}" 
                       for i, m in enumerate(sorted(memories, key=lambda x: x.created_at), 1)]
        
        user_prompt = f"""合并以下{len(memories)}条记忆：

{chr(10).join(memory_list)}

输出合并后的记忆："""

        merged = self._call_llm(system_prompt, user_prompt)
        
        if not merged:
            contents = [m.memory for m in sorted(memories, key=lambda x: x.created_at)]
            merged = "综合记忆：" + "；".join(contents)
        
        return merged
    
    def should_merge(self, memory_a: MemoryNode, memory_b: MemoryNode) -> Dict:
        """
        使用LLM判断两条记忆是否应该合并
        """
        system_prompt = """你是记忆管理专家。判断两条记忆是否应该合并。

合并条件：
1. 描述同一主题或同类信息
2. 没有逻辑冲突
3. 合并后信息更完整

输出JSON：
{"should_merge": true/false, "reason": "理由", "merge_type": "complement/update/duplicate"}

merge_type: complement(互补)、update(更新)、duplicate(去重)

只输出JSON。"""

        user_prompt = f"""判断是否应合并：

记忆A：{memory_a.key} - {memory_a.memory}（标签：{', '.join(memory_a.tags)}）

记忆B：{memory_b.key} - {memory_b.memory}（标签：{', '.join(memory_b.tags)}）"""

        response = self._call_llm(system_prompt, user_prompt)
        result = self._parse_json_response(response)
        return result or {"should_merge": False, "reason": "解析失败"}
    
    def resolve_conflict(self, memory_a: MemoryNode, memory_b: MemoryNode) -> Dict:
        """
        使用LLM解决记忆冲突
        """
        system_prompt = """你是记忆冲突仲裁专家。根据以下规则解决冲突：

规则（按优先级）：
1. 用户明确陈述 > 系统推断 > 第三方来源
2. 来源可信度相近时，以最新记忆为准
3. 可互补时（如：平时喜欢A，特殊情况需要B），建议合并

输出JSON：
{
    "action": "keep_a/keep_b/merge/coexist",
    "reason": "解决理由",
    "merged_content": "如果merge，提供合并内容",
    "confidence": 0.0-1.0
}

action: keep_a(保留A废弃B)、keep_b(保留B废弃A)、merge(合并)、coexist(共存降权)

只输出JSON。"""

        user_prompt = f"""解决以下冲突记忆：

记忆A：{memory_a.memory}
- 更新时间：{memory_a.updated_at}
- 来源：{memory_a.source_type}（可信度：{memory_a.source_credibility}）

记忆B：{memory_b.memory}
- 更新时间：{memory_b.updated_at}
- 来源：{memory_b.source_type}（可信度：{memory_b.source_credibility}）"""

        response = self._call_llm(system_prompt, user_prompt)
        result = self._parse_json_response(response)
        
        if result:
            action_map = {
                "keep_a": {"action": UpdateAction.OVERWRITE, "keep_id": memory_a.id, "deprecate_id": memory_b.id},
                "keep_b": {"action": UpdateAction.OVERWRITE, "keep_id": memory_b.id, "deprecate_id": memory_a.id},
                "merge": {"action": UpdateAction.MERGE, "memories": [memory_a.id, memory_b.id], 
                         "merged_content": result.get("merged_content", "")},
                "coexist": {"action": "coexist", 
                           "newer_id": memory_a.id if memory_a.updated_at > memory_b.updated_at else memory_b.id}
            }
            action = result.get("action", "keep_b")
            resolution = action_map.get(action, action_map["keep_b"])
            resolution["reason"] = result.get("reason", "LLM决策")
            resolution["confidence"] = result.get("confidence", 0.8)
            return resolution
        
        # 默认：时效性优先
        if memory_a.updated_at > memory_b.updated_at:
            return {"action": UpdateAction.OVERWRITE, "keep_id": memory_a.id, 
                    "deprecate_id": memory_b.id, "reason": "默认：时效性优先"}
        return {"action": UpdateAction.OVERWRITE, "keep_id": memory_b.id, 
                "deprecate_id": memory_a.id, "reason": "默认：时效性优先"}
    
    def determine_update_action(self, new_memory: MemoryNode, 
                                 existing_memory: MemoryNode) -> Dict:
        """
        使用LLM判断新记忆应该如何处理现有记忆
        """
        system_prompt = """你是记忆更新决策专家。判断新记忆如何处理现有记忆。

操作类型：
1. OVERWRITE（覆盖）：完全替代，适用于纠错或明确更新
2. MERGE（合并）：融合信息，适用于互补
3. VERSION（版本化）：保留历史创建新版本，适用于需追溯场景
4. IGNORE（忽略）：不影响现有记忆，适用于重复或无效信息
5. CONFLICT（冲突）：需进一步消解，适用于明显矛盾

输出JSON：
{"action": "OVERWRITE/MERGE/VERSION/IGNORE/CONFLICT", "reason": "理由", "confidence": 0.0-1.0}

只输出JSON。"""

        user_prompt = f"""判断新记忆如何处理现有记忆：

新记忆：{new_memory.key} - {new_memory.memory}（来源：{new_memory.source_type}）

现有记忆：{existing_memory.key} - {existing_memory.memory}（来源：{existing_memory.source_type}）"""

        response = self._call_llm(system_prompt, user_prompt)
        result = self._parse_json_response(response)
        
        if result:
            action_map = {
                "OVERWRITE": UpdateAction.OVERWRITE,
                "MERGE": UpdateAction.MERGE,
                "VERSION": UpdateAction.VERSION,
                "CONFLICT": UpdateAction.CONFLICT,
                "IGNORE": None
            }
            return {
                "action": action_map.get(result.get("action", "IGNORE")),
                "reason": result.get("reason", ""),
                "confidence": result.get("confidence", 0.5)
            }
        return {"action": None, "reason": "解析失败", "confidence": 0.0}


# 创建全局LLM服务实例
llm_service = LLMService()
print(f"[LLMService] 已初始化，模型: {llm_service.model}")


[LLMService] 已初始化，模型: gpt-4.1-mini


In [5]:
# ============================================================
# 【核心示例】记忆合并模块 - Memory Merge Module
# ============================================================
# 本模块可单独提取作为书本示例代码

class MemoryMerger:
    """
    记忆合并器：将多条相似或互补的记忆合并为一条
    
    适用场景：
    1. 用户多次表达同一主题的偏好（如多次提到喜欢不同水果）
    2. 同一事实的多个片段需要整合
    3. 重复记忆的去重与压缩
    
    合并策略：
    - 内容合并：将多条记忆的内容整合为一条综合描述
    - 标签合并：取所有记忆标签的并集
    - 置信度合并：取最高置信度并可选择性提升
    - 时间戳处理：保留最早创建时间，更新为当前时间
    """
    
    def __init__(self, config: MemoryUpdateConfig = None):
        self.config = config or MemoryUpdateConfig()
    
    def should_merge(self, memory_a: MemoryNode, memory_b: MemoryNode, 
                     similarity: float) -> bool:
        """
        判断两条记忆是否应该合并（使用LLM智能判断）
        
        Args:
            memory_a: 记忆A
            memory_b: 记忆B
            similarity: 相似度分数 (0-1)
            
        Returns:
            是否应该合并
        """
        # 基本条件：相似度超过阈值
        if similarity < self.config.similarity_threshold:
            return False
        
        # 必须是同一用户的记忆
        if memory_a.user_id != memory_b.user_id:
            return False
        
        # 必须是同一类型的记忆
        if memory_a.type != memory_b.type:
            return False
        
        # 使用LLM判断是否应该合并
        merge_decision = llm_service.should_merge(memory_a, memory_b)
        
        if self.config.verbose:
            print(f"[MemoryMerger] LLM合并判断: {merge_decision}")
        
        # 如果LLM判断不应合并，检查是否是冲突
        if not merge_decision.get("should_merge", False):
            # 检查是否存在冲突
            conflict = llm_service.detect_conflict(memory_a, memory_b)
            if conflict:
                if self.config.verbose:
                    print(f"[MemoryMerger] 检测到冲突，不合并: {conflict.get('description')}")
            return False
        
        return True
    
    def merge_two(self, memory_a: MemoryNode, memory_b: MemoryNode) -> MemoryNode:
        """
        合并两条记忆为一条新记忆
        
        Args:
            memory_a: 记忆A
            memory_b: 记忆B
            
        Returns:
            合并后的新记忆节点
        """
        now = datetime.now().isoformat()
        
        # 使用LLM进行智能合并
        merged_content = llm_service.merge_memories([memory_a, memory_b])
        
        # 合并标签（取并集）
        merged_tags = list(set(memory_a.tags + memory_b.tags))
        
        # 合并关键词（取较长的或组合）
        merged_key = memory_a.key if len(memory_a.key) >= len(memory_b.key) else memory_b.key
        
        # 置信度：取最高值并可选提升
        merged_confidence = min(
            max(memory_a.confidence, memory_b.confidence) + self.config.merge_confidence_boost,
            1.0
        )
        
        # 来源可信度：取最高值
        merged_credibility = max(memory_a.source_credibility, memory_b.source_credibility)
        
        # 访问次数：累加
        merged_access_count = memory_a.access_count + memory_b.access_count
        
        # 创建合并后的新记忆
        merged_memory = MemoryNode(
            id=str(uuid.uuid4()),
            key=merged_key,
            memory=merged_content,
            tags=merged_tags,
            type=memory_a.type,
            confidence=merged_confidence,
            created_at=min(memory_a.created_at, memory_b.created_at),  # 保留最早时间
            updated_at=now,
            user_id=memory_a.user_id,
            session_id=memory_a.session_id,
            status=MemoryStatus.ACTIVATED.value,
            source_type="merged",
            source_credibility=merged_credibility,
            access_count=merged_access_count,
            last_accessed=now,
            decay_score=1.0,  # 新合并的记忆重置衰减分数
            version=1,
            parent_version_id=None
        )
        
        return merged_memory
    
    def merge_batch(self, memories: List[MemoryNode]) -> Tuple[MemoryNode, List[str]]:
        """
        批量合并多条记忆
        
        Args:
            memories: 待合并的记忆列表
            
        Returns:
            (合并后的记忆, 被合并的原记忆ID列表)
        """
        if not memories:
            raise ValueError("记忆列表不能为空")
        
        if len(memories) == 1:
            return memories[0], []
        
        now = datetime.now().isoformat()
        
        # 使用LLM合并所有内容
        merged_content = llm_service.merge_memories(memories)
        
        # 合并所有标签
        all_tags = []
        for m in memories:
            all_tags.extend(m.tags)
        merged_tags = list(set(all_tags))
        
        # 选择最具代表性的key（最长的）
        merged_key = max(memories, key=lambda m: len(m.key)).key
        
        # 置信度和可信度
        merged_confidence = min(
            max(m.confidence for m in memories) + self.config.merge_confidence_boost,
            1.0
        )
        merged_credibility = max(m.source_credibility for m in memories)
        
        # 累计访问次数
        total_access = sum(m.access_count for m in memories)
        
        # 创建合并记忆
        merged_memory = MemoryNode(
            id=str(uuid.uuid4()),
            key=merged_key,
            memory=merged_content,
            tags=merged_tags,
            type=memories[0].type,
            confidence=merged_confidence,
            created_at=min(m.created_at for m in memories),
            updated_at=now,
            user_id=memories[0].user_id,
            session_id=memories[0].session_id,
            status=MemoryStatus.ACTIVATED.value,
            source_type="merged",
            source_credibility=merged_credibility,
            access_count=total_access,
            last_accessed=now,
            decay_score=1.0,
            version=1,
            parent_version_id=None
        )
        
        original_ids = [m.id for m in memories]
        
        if self.config.verbose:
            print(f"[MemoryMerger] 合并了 {len(memories)} 条记忆")
            print(f"  原记忆IDs: {original_ids}")
            print(f"  新记忆ID: {merged_memory.id}")
            print(f"  合并内容: {merged_content[:100]}...")
        
        return merged_memory, original_ids


# ============================================================
# 记忆合并示例函数（可直接用于书本示例）
# ============================================================

def example_memory_merge():
    """
    记忆合并示例：展示如何合并用户的多条水果偏好记忆
    """
    # 创建示例记忆
    memories = [
        MemoryNode(
            id="m1",
            key="水果偏好",
            memory="用户在2026年1月表达了对于苹果的喜爱",
            tags=["偏好", "水果", "苹果"],
            type=MemoryType.PREFERENCE.value,
            confidence=0.9,
            created_at="2026-01-05T10:00:00",
            updated_at="2026-01-05T10:00:00",
            user_id="user_001",
            session_id="session_001"
        ),
        MemoryNode(
            id="m2",
            key="水果偏好",
            memory="用户在2026年1月表达了对于香蕉的喜欢",
            tags=["偏好", "水果", "香蕉"],
            type=MemoryType.PREFERENCE.value,
            confidence=0.85,
            created_at="2026-01-06T14:00:00",
            updated_at="2026-01-06T14:00:00",
            user_id="user_001",
            session_id="session_002"
        ),
        MemoryNode(
            id="m3",
            key="水果偏好",
            memory="用户在2026年1月说了橘子和榴莲很好吃",
            tags=["偏好", "水果", "橘子", "榴莲"],
            type=MemoryType.PREFERENCE.value,
            confidence=0.88,
            created_at="2026-01-07T09:00:00",
            updated_at="2026-01-07T09:00:00",
            user_id="user_001",
            session_id="session_003"
        )
    ]
    
    # 执行合并
    merger = MemoryMerger(MemoryUpdateConfig(verbose=True))
    merged_memory, original_ids = merger.merge_batch(memories)
    
    print("\n" + "="*60)
    print("合并结果:")
    print("="*60)
    print(f"新记忆ID: {merged_memory.id}")
    print(f"内容: {merged_memory.memory}")
    print(f"标签: {merged_memory.tags}")
    print(f"置信度: {merged_memory.confidence}")
    print(f"创建时间: {merged_memory.created_at}")
    print(f"更新时间: {merged_memory.updated_at}")
    
    return merged_memory

# 运行示例
print("【记忆合并示例】")
example_memory_merge()


【记忆合并示例】
[LLMService] API调用失败: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-api*****here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
[MemoryMerger] 合并了 3 条记忆
  原记忆IDs: ['m1', 'm2', 'm3']
  新记忆ID: 7b062f64-e9b4-4b86-b065-2bb7313cce79
  合并内容: 综合记忆：用户在2026年1月表达了对于苹果的喜爱；用户在2026年1月表达了对于香蕉的喜欢；用户在2026年1月说了橘子和榴莲很好吃...

合并结果:
新记忆ID: 7b062f64-e9b4-4b86-b065-2bb7313cce79
内容: 综合记忆：用户在2026年1月表达了对于苹果的喜爱；用户在2026年1月表达了对于香蕉的喜欢；用户在2026年1月说了橘子和榴莲很好吃
标签: ['苹果', '香蕉', '橘子', '榴莲', '水果', '偏好']
置信度: 1.0
创建时间: 2026-01-05T10:00:00
更新时间: 2026-01-11T21:26:23.942095


MemoryNode(id='7b062f64-e9b4-4b86-b065-2bb7313cce79', key='水果偏好', memory='综合记忆：用户在2026年1月表达了对于苹果的喜爱；用户在2026年1月表达了对于香蕉的喜欢；用户在2026年1月说了橘子和榴莲很好吃', tags=['苹果', '香蕉', '橘子', '榴莲', '水果', '偏好'], type='preference', confidence=1.0, created_at='2026-01-05T10:00:00', updated_at='2026-01-11T21:26:23.942095', user_id='user_001', session_id='session_001', status='activated', source_type='merged', source_credibility=1.0, access_count=0, last_accessed='2026-01-11T21:26:23.942095', decay_score=1.0, version=1, parent_version_id=None, embedding=None)

In [6]:
# ============================================================
# 记忆覆盖模块 - Memory Overwrite Module
# ============================================================

class MemoryOverwriter:
    """
    记忆覆盖器：用新信息完全替代旧信息
    
    适用场景：
    1. 用户明确修改偏好：以后不要再推荐咖啡，改成茶
    2. 配置项被替换：默认模型从A切换到B
    3. 明确纠错且旧信息无保留价值：我的生日不是3月2日，是3月12日
    """
    
    def __init__(self, config: MemoryUpdateConfig = None):
        self.config = config or MemoryUpdateConfig()
    
    def overwrite(self, old_memory: MemoryNode, new_content: str, 
                  new_key: Optional[str] = None,
                  new_tags: Optional[List[str]] = None,
                  reason: str = "用户主动更新") -> MemoryNode:
        """
        覆盖记忆内容
        
        Args:
            old_memory: 原记忆
            new_content: 新内容
            new_key: 新关键词（可选）
            new_tags: 新标签（可选）
            reason: 覆盖原因
            
        Returns:
            更新后的记忆节点
        """
        now = datetime.now().isoformat()
        
        # 创建覆盖后的记忆（保留ID，更新内容）
        updated_memory = MemoryNode(
            id=old_memory.id,  # 保留原ID
            key=new_key or old_memory.key,
            memory=new_content,
            tags=new_tags or old_memory.tags,
            type=old_memory.type,
            confidence=old_memory.confidence,  # 保留置信度
            created_at=old_memory.created_at,  # 保留创建时间
            updated_at=now,  # 更新时间
            user_id=old_memory.user_id,
            session_id=old_memory.session_id,
            status=MemoryStatus.ACTIVATED.value,
            source_type=old_memory.source_type,
            source_credibility=old_memory.source_credibility,
            access_count=old_memory.access_count,
            last_accessed=now,
            decay_score=1.0,  # 重置衰减分数
            version=old_memory.version + 1,  # 版本号+1
            parent_version_id=old_memory.id
        )
        
        if self.config.verbose:
            print(f"[MemoryOverwriter] 覆盖记忆 {old_memory.id}")
            print(f"  原内容: {old_memory.memory[:50]}...")
            print(f"  新内容: {new_content[:50]}...")
            print(f"  原因: {reason}")
        
        return updated_memory
    
    def deprecate(self, memory: MemoryNode, reason: str = "信息过时") -> MemoryNode:
        """
        将记忆标记为废弃（不删除，但不再参与检索）
        
        Args:
            memory: 待废弃的记忆
            reason: 废弃原因
            
        Returns:
            更新后的记忆节点
        """
        deprecated_memory = copy.deepcopy(memory)
        deprecated_memory.status = MemoryStatus.DEPRECATED.value
        deprecated_memory.updated_at = datetime.now().isoformat()
        
        if self.config.verbose:
            print(f"[MemoryOverwriter] 废弃记忆 {memory.id}")
            print(f"  内容: {memory.memory[:50]}...")
            print(f"  原因: {reason}")
        
        return deprecated_memory


# ============================================================
# 记忆版本化模块 - Memory Versioning Module
# ============================================================

class MemoryVersionManager:
    """
    记忆版本管理器：在不删除历史的情况下记录记忆的变化轨迹
    
    适用场景：
    1. 企业文档知识库的版本迭代
    2. 需要审计追溯的场景
    3. 误更新后需要回滚的场景
    """
    
    def __init__(self, config: MemoryUpdateConfig = None):
        self.config = config or MemoryUpdateConfig()
        # 版本链存储：memory_id -> VersionedMemory
        self.version_registry: Dict[str, VersionedMemory] = {}
    
    def create_version(self, old_memory: MemoryNode, new_content: str,
                       change_description: str) -> Tuple[MemoryNode, MemoryNode]:
        """
        创建新版本（保留旧版本）
        
        Args:
            old_memory: 原记忆
            new_content: 新内容
            change_description: 变更描述
            
        Returns:
            (归档的旧版本, 新版本)
        """
        now = datetime.now().isoformat()
        
        # 1. 将旧版本归档
        archived_old = copy.deepcopy(old_memory)
        archived_old.status = MemoryStatus.ARCHIVED.value
        archived_old.updated_at = now
        
        # 2. 创建新版本
        new_version = MemoryNode(
            id=str(uuid.uuid4()),  # 新ID
            key=old_memory.key,
            memory=new_content,
            tags=old_memory.tags,
            type=old_memory.type,
            confidence=old_memory.confidence,
            created_at=now,  # 新版本的创建时间
            updated_at=now,
            user_id=old_memory.user_id,
            session_id=old_memory.session_id,
            status=MemoryStatus.ACTIVATED.value,
            source_type=old_memory.source_type,
            source_credibility=old_memory.source_credibility,
            access_count=0,  # 新版本重置访问计数
            last_accessed=None,
            decay_score=1.0,
            version=old_memory.version + 1,
            parent_version_id=old_memory.id  # 指向父版本
        )
        
        # 3. 更新版本注册表
        root_id = self._find_root_id(old_memory.id)
        if root_id not in self.version_registry:
            self.version_registry[root_id] = VersionedMemory(
                memory_id=root_id,
                version_chain=[old_memory.id],
                current_version=old_memory.version
            )
        
        versioned = self.version_registry[root_id]
        versioned.version_chain.append(new_version.id)
        versioned.current_version = new_version.version
        versioned.change_log.append({
            "from_version": old_memory.version,
            "to_version": new_version.version,
            "from_id": old_memory.id,
            "to_id": new_version.id,
            "description": change_description,
            "timestamp": now
        })
        
        if self.config.verbose:
            print(f"[MemoryVersionManager] 创建新版本")
            print(f"  旧版本ID: {old_memory.id} (v{old_memory.version}) -> 已归档")
            print(f"  新版本ID: {new_version.id} (v{new_version.version})")
            print(f"  变更描述: {change_description}")
        
        return archived_old, new_version
    
    def _find_root_id(self, memory_id: str) -> str:
        """查找版本链的根ID"""
        for root_id, versioned in self.version_registry.items():
            if memory_id in versioned.version_chain:
                return root_id
        return memory_id  # 如果不在任何链中，自身就是根
    
    def get_version_history(self, memory_id: str) -> Optional[VersionedMemory]:
        """获取记忆的版本历史"""
        root_id = self._find_root_id(memory_id)
        return self.version_registry.get(root_id)
    
    def rollback(self, memory_id: str, target_version: int, 
                 memory_store: Dict[str, MemoryNode]) -> Optional[MemoryNode]:
        """
        回滚到指定版本
        
        Args:
            memory_id: 当前记忆ID
            target_version: 目标版本号
            memory_store: 记忆存储
            
        Returns:
            回滚后激活的记忆节点
        """
        versioned = self.get_version_history(memory_id)
        if not versioned:
            print(f"[MemoryVersionManager] 未找到版本历史: {memory_id}")
            return None
        
        # 查找目标版本
        for vid in versioned.version_chain:
            if vid in memory_store:
                mem = memory_store[vid]
                if mem.version == target_version:
                    # 激活目标版本
                    mem.status = MemoryStatus.ACTIVATED.value
                    mem.updated_at = datetime.now().isoformat()
                    
                    # 归档当前版本
                    current_id = versioned.version_chain[-1]
                    if current_id in memory_store and current_id != vid:
                        memory_store[current_id].status = MemoryStatus.ARCHIVED.value
                    
                    if self.config.verbose:
                        print(f"[MemoryVersionManager] 回滚到版本 {target_version}")
                        print(f"  激活记忆ID: {vid}")
                    
                    return mem
        
        print(f"[MemoryVersionManager] 未找到版本 {target_version}")
        return None


# ============================================================
# 记忆覆盖与版本化示例
# ============================================================

def example_memory_overwrite_and_version():
    """
    记忆覆盖与版本化示例
    """
    print("\n" + "="*60)
    print("【记忆覆盖示例】")
    print("="*60)
    
    # 原始记忆：用户生日
    original_memory = MemoryNode(
        id="birthday_001",
        key="用户生日",
        memory="用户的生日是3月2日",
        tags=["个人信息", "生日"],
        type=MemoryType.FACT.value,
        confidence=0.9,
        created_at="2025-06-01T10:00:00",
        updated_at="2025-06-01T10:00:00",
        user_id="user_001",
        session_id="session_001"
    )
    
    # 用户纠错：覆盖
    overwriter = MemoryOverwriter(MemoryUpdateConfig(verbose=True))
    updated_memory = overwriter.overwrite(
        old_memory=original_memory,
        new_content="用户的生日是3月12日",
        reason="用户明确纠错"
    )
    
    print(f"\n更新后版本号: {updated_memory.version}")
    
    print("\n" + "="*60)
    print("【记忆版本化示例】")
    print("="*60)
    
    # 企业政策记忆
    policy_memory = MemoryNode(
        id="policy_001",
        key="报销流程",
        memory="报销流程需要纸质签字后扫描上传",
        tags=["政策", "报销"],
        type=MemoryType.FACT.value,
        confidence=0.95,
        created_at="2024-01-01T10:00:00",
        updated_at="2024-01-01T10:00:00",
        user_id="system",
        session_id="system",
        source_type="official_doc",
        source_credibility=0.9
    )
    
    # 政策更新：创建新版本
    version_manager = MemoryVersionManager(MemoryUpdateConfig(verbose=True))
    archived, new_version = version_manager.create_version(
        old_memory=policy_memory,
        new_content="报销流程已升级为全线上电子签名，无需纸质材料",
        change_description="2025年政策升级：全面数字化"
    )
    
    print(f"\n归档版本状态: {archived.status}")
    print(f"新版本状态: {new_version.status}")
    print(f"版本历史: {version_manager.get_version_history(policy_memory.id)}")

# 运行示例
example_memory_overwrite_and_version()



【记忆覆盖示例】
[MemoryOverwriter] 覆盖记忆 birthday_001
  原内容: 用户的生日是3月2日...
  新内容: 用户的生日是3月12日...
  原因: 用户明确纠错

更新后版本号: 2

【记忆版本化示例】
[MemoryVersionManager] 创建新版本
  旧版本ID: policy_001 (v1) -> 已归档
  新版本ID: 44a31275-802c-475d-9b69-c0c6fe34dde4 (v2)
  变更描述: 2025年政策升级：全面数字化

归档版本状态: archived
新版本状态: activated
版本历史: VersionedMemory(memory_id='policy_001', version_chain=['policy_001', '44a31275-802c-475d-9b69-c0c6fe34dde4'], current_version=2, change_log=[{'from_version': 1, 'to_version': 2, 'from_id': 'policy_001', 'to_id': '44a31275-802c-475d-9b69-c0c6fe34dde4', 'description': '2025年政策升级：全面数字化', 'timestamp': '2026-01-11T21:26:26.407560'}])


In [7]:
# ============================================================
# 冲突消解模块 - Conflict Resolution Module
# ============================================================

class ConflictResolver:
    """
    冲突消解器：检测并解决记忆之间的冲突
    
    冲突类型：
    1. 显式事实冲突：涉及具体数值、日期或身份的硬性矛盾
    2. 隐式偏好冲突：用户行为表现出的长期倾向与短期行为的偏离
    
    消解策略：
    1. 基于时效性：最新信息优先
    2. 基于来源可信度：高可信度来源优先
    3. 概率性共存：降低旧记忆权重但不删除
    """
    
    def __init__(self, config: MemoryUpdateConfig = None):
        self.config = config or MemoryUpdateConfig()
        self.conflict_records: List[ConflictRecord] = []
    
    def detect_conflict(self, new_memory: MemoryNode, 
                       existing_memories: List[MemoryNode]) -> List[ConflictRecord]:
        """
        检测新记忆与现有记忆之间的冲突
        
        Args:
            new_memory: 新入库的记忆
            existing_memories: 现有相关记忆列表
            
        Returns:
            检测到的冲突记录列表
        """
        conflicts = []
        
        for existing in existing_memories:
            # 跳过非活跃状态的记忆
            if existing.status != MemoryStatus.ACTIVATED.value:
                continue
            
            # 使用LLM检测冲突
            conflict_info = llm_service.detect_conflict(new_memory, existing)
            
            if conflict_info and conflict_info.get("has_conflict"):
                record = ConflictRecord(
                    conflict_id=str(uuid.uuid4()),
                    memory_id_a=new_memory.id,
                    memory_id_b=existing.id,
                    conflict_type=conflict_info["conflict_type"],
                    description=conflict_info["description"]
                )
                conflicts.append(record)
                self.conflict_records.append(record)
                
                if self.config.verbose:
                    print(f"[ConflictResolver] 检测到冲突:")
                    print(f"  记忆A: {new_memory.memory[:40]}...")
                    print(f"  记忆B: {existing.memory[:40]}...")
                    print(f"  类型: {conflict_info['conflict_type']}")
        
        return conflicts
    
    def resolve_by_time_priority(self, memory_a: MemoryNode, 
                                  memory_b: MemoryNode) -> Dict:
        """
        基于时效性解决冲突：保留最新的记忆
        
        Args:
            memory_a: 记忆A
            memory_b: 记忆B
            
        Returns:
            解决方案
        """
        # 比较更新时间
        time_a = datetime.fromisoformat(memory_a.updated_at.replace('Z', '+00:00'))
        time_b = datetime.fromisoformat(memory_b.updated_at.replace('Z', '+00:00'))
        
        if time_a >= time_b:
            winner, loser = memory_a, memory_b
        else:
            winner, loser = memory_b, memory_a
        
        resolution = {
            "strategy": "time_priority",
            "action": UpdateAction.OVERWRITE,
            "keep_id": winner.id,
            "deprecate_id": loser.id,
            "reason": f"时效性优先：保留更新时间较新的记忆 ({winner.updated_at} > {loser.updated_at})"
        }
        
        if self.config.verbose:
            print(f"[ConflictResolver] 时效性解决:")
            print(f"  保留: {winner.id} ({winner.updated_at})")
            print(f"  废弃: {loser.id} ({loser.updated_at})")
        
        return resolution
    
    def resolve_by_source_priority(self, memory_a: MemoryNode, 
                                    memory_b: MemoryNode) -> Dict:
        """
        基于来源可信度解决冲突：保留可信度更高的记忆
        
        来源可信度排序（从高到低）：
        1. 用户明确陈述 (1.0)
        2. 官方文档 (0.9)
        3. 系统推断 (0.7)
        4. 第三方来源 (0.5)
        """
        if memory_a.source_credibility >= memory_b.source_credibility:
            winner, loser = memory_a, memory_b
        else:
            winner, loser = memory_b, memory_a
        
        resolution = {
            "strategy": "source_priority",
            "action": UpdateAction.OVERWRITE,
            "keep_id": winner.id,
            "deprecate_id": loser.id,
            "reason": f"来源优先：保留可信度更高的记忆 ({winner.source_credibility} > {loser.source_credibility})"
        }
        
        if self.config.verbose:
            print(f"[ConflictResolver] 来源可信度解决:")
            print(f"  保留: {winner.id} (可信度: {winner.source_credibility})")
            print(f"  废弃: {loser.id} (可信度: {loser.source_credibility})")
        
        return resolution
    
    def resolve_by_coexistence(self, memory_a: MemoryNode, 
                                memory_b: MemoryNode) -> Dict:
        """
        概率性共存：降低旧记忆的权重但不删除
        
        适用于偏好类冲突，保留记忆的弹性
        """
        # 确定哪个是较新的
        time_a = datetime.fromisoformat(memory_a.updated_at.replace('Z', '+00:00'))
        time_b = datetime.fromisoformat(memory_b.updated_at.replace('Z', '+00:00'))
        
        if time_a >= time_b:
            newer, older = memory_a, memory_b
        else:
            newer, older = memory_b, memory_a
        
        # 降低旧记忆的衰减分数
        decay_factor = 0.5
        
        resolution = {
            "strategy": "coexistence",
            "action": "weight_adjustment",
            "newer_id": newer.id,
            "older_id": older.id,
            "older_decay_factor": decay_factor,
            "reason": f"概率共存：保留两条记忆，降低旧记忆权重至 {decay_factor}"
        }
        
        if self.config.verbose:
            print(f"[ConflictResolver] 概率共存解决:")
            print(f"  较新记忆: {newer.id} (保持权重)")
            print(f"  较旧记忆: {older.id} (权重降至 {decay_factor})")
        
        return resolution
    
    def auto_resolve(self, conflict: ConflictRecord, 
                     memory_store: Dict[str, MemoryNode]) -> Dict:
        """
        使用LLM自动解决冲突
        
        Args:
            conflict: 冲突记录
            memory_store: 记忆存储
            
        Returns:
            解决方案
        """
        memory_a = memory_store.get(conflict.memory_id_a)
        memory_b = memory_store.get(conflict.memory_id_b)
        
        if not memory_a or not memory_b:
            return {"error": "记忆不存在"}
        
        # 使用LLM智能解决冲突
        resolution = llm_service.resolve_conflict(memory_a, memory_b)
        
        if self.config.verbose:
            print(f"[ConflictResolver] LLM解决方案: {resolution.get('reason', 'N/A')}")
        
        return resolution


# ============================================================
# 冲突消解示例
# ============================================================

def example_conflict_resolution():
    """
    冲突消解示例：展示不同类型冲突的处理
    """
    print("\n" + "="*60)
    print("【冲突消解示例】")
    print("="*60)
    
    config = MemoryUpdateConfig(verbose=True)
    resolver = ConflictResolver(config)
    
    # 示例1：事实冲突 - 用户饮食偏好
    print("\n--- 示例1：事实冲突（饮食偏好）---")
    memory_old = MemoryNode(
        id="diet_001",
        key="饮食偏好",
        memory="用户喜欢吃辣，常去四川餐馆",
        tags=["偏好", "饮食"],
        type=MemoryType.PREFERENCE.value,
        confidence=0.85,
        created_at="2024-05-01T10:00:00",
        updated_at="2024-05-01T10:00:00",
        user_id="user_001",
        session_id="session_001",
        source_type="system_inference",
        source_credibility=0.7
    )
    
    memory_new = MemoryNode(
        id="diet_002",
        key="饮食偏好",
        memory="用户反馈最近胃部不适，需要饮食清淡，不喜欢辛辣",
        tags=["偏好", "饮食", "健康"],
        type=MemoryType.PREFERENCE.value,
        confidence=0.95,
        created_at="2026-01-08T14:00:00",
        updated_at="2026-01-08T14:00:00",
        user_id="user_001",
        session_id="session_010",
        source_type="user_explicit",
        source_credibility=1.0
    )
    
    # 检测冲突
    conflicts = resolver.detect_conflict(memory_new, [memory_old])
    
    if conflicts:
        # 自动解决
        memory_store = {memory_old.id: memory_old, memory_new.id: memory_new}
        resolution = resolver.auto_resolve(conflicts[0], memory_store)
        print(f"\n解决方案: {resolution}")
    
    # 示例2：偏好冲突 - 消费习惯
    print("\n--- 示例2：偏好冲突（消费习惯）---")
    memory_long_term = MemoryNode(
        id="consume_001",
        key="消费习惯",
        memory="用户一贯坚持极简主义消费，喜欢性价比高的产品",
        tags=["偏好", "消费"],
        type=MemoryType.PREFERENCE.value,
        confidence=0.9,
        created_at="2024-01-01T10:00:00",
        updated_at="2024-06-01T10:00:00",
        user_id="user_002",
        session_id="session_001",
        source_type="system_inference",
        source_credibility=0.8
    )
    
    memory_recent = MemoryNode(
        id="consume_002",
        key="消费习惯",
        memory="用户最近频繁咨询高端奢侈品，不喜欢便宜货",
        tags=["偏好", "消费", "奢侈品"],
        type=MemoryType.PREFERENCE.value,
        confidence=0.75,
        created_at="2026-01-05T10:00:00",
        updated_at="2026-01-05T10:00:00",
        user_id="user_002",
        session_id="session_020",
        source_type="system_inference",
        source_credibility=0.7
    )
    
    # 这是偏好冲突，应该采用概率共存
    resolution = resolver.resolve_by_coexistence(memory_long_term, memory_recent)
    print(f"\n解决方案: {resolution}")

# 运行示例
example_conflict_resolution()



【冲突消解示例】

--- 示例1：事实冲突（饮食偏好）---
[LLMService] API调用失败: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-api*****here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
[LLMService] JSON解析失败: Expecting value: line 1 column 1 (char 0)

--- 示例2：偏好冲突（消费习惯）---
[ConflictResolver] 概率共存解决:
  较新记忆: consume_002 (保持权重)
  较旧记忆: consume_001 (权重降至 0.5)

解决方案: {'strategy': 'coexistence', 'action': 'weight_adjustment', 'newer_id': 'consume_002', 'older_id': 'consume_001', 'older_decay_factor': 0.5, 'reason': '概率共存：保留两条记忆，降低旧记忆权重至 0.5'}


In [8]:
# ============================================================
# 遗忘策略模块 - Forgetting Strategy Module
# ============================================================

class MemoryForgetter:
    """
    记忆遗忘器：实现工程化的遗忘机制
    
    遗忘不是"记忆丢失"的缺陷，而是维持系统长期稳定性的必要能力。
    
    四种遗忘动作（风险从低到高）：
    1. 降权：降低检索优先级，但保留以备审计
    2. 归档：从在线索引迁移到冷存储
    3. 压缩：将多条相似记忆汇总为摘要
    4. 删除：彻底移除（仅用于隐私合规或明显垃圾数据）
    
    衰减机制：
    - 类比艾宾浩斯遗忘曲线
    - 记忆的保留概率与时间、复用、巩固强相关
    """
    
    def __init__(self, config: MemoryUpdateConfig = None):
        self.config = config or MemoryUpdateConfig()
    
    def calculate_decay(self, memory: MemoryNode, 
                        current_time: Optional[datetime] = None) -> float:
        """
        计算记忆的衰减分数
        
        衰减公式（类似艾宾浩斯曲线）：
        decay_score = e^(-decay_rate * time_elapsed_hours / 24) * (1 + log(1 + access_count))
        
        Args:
            memory: 记忆节点
            current_time: 当前时间（默认为now）
            
        Returns:
            衰减后的分数 (0-1)
        """
        if current_time is None:
            current_time = datetime.now()
        
        # 计算距离最后访问/更新的时间
        last_time_str = memory.last_accessed or memory.updated_at
        try:
            last_time = datetime.fromisoformat(last_time_str.replace('Z', '+00:00'))
            if last_time.tzinfo:
                last_time = last_time.replace(tzinfo=None)
        except:
            last_time = current_time
        
        # 时间衰减（小时）
        hours_elapsed = (current_time - last_time).total_seconds() / 3600
        
        # 基础衰减
        time_decay = math.exp(-self.config.decay_rate * hours_elapsed / 24)
        
        # 访问次数加成（复用强化）
        access_boost = 1 + math.log(1 + memory.access_count)
        
        # 最终衰减分数
        decay_score = min(time_decay * access_boost, 1.0)
        decay_score = max(decay_score, self.config.min_decay_score)
        
        return decay_score
    
    def update_decay_scores(self, memories: List[MemoryNode]) -> List[MemoryNode]:
        """
        批量更新记忆的衰减分数
        
        Args:
            memories: 记忆列表
            
        Returns:
            更新后的记忆列表
        """
        current_time = datetime.now()
        updated = []
        
        for memory in memories:
            if memory.status == MemoryStatus.ACTIVATED.value:
                memory.decay_score = self.calculate_decay(memory, current_time)
                updated.append(memory)
                
                if self.config.verbose and memory.decay_score < 0.5:
                    print(f"[MemoryForgetter] 低衰减分数: {memory.id}")
                    print(f"  内容: {memory.memory[:30]}...")
                    print(f"  衰减分数: {memory.decay_score:.3f}")
        
        return updated
    
    def downweight(self, memory: MemoryNode, factor: float = 0.5) -> MemoryNode:
        """
        降权：降低记忆的检索优先级
        
        Args:
            memory: 记忆节点
            factor: 降权因子
            
        Returns:
            降权后的记忆
        """
        memory.decay_score *= factor
        memory.decay_score = max(memory.decay_score, self.config.min_decay_score)
        memory.updated_at = datetime.now().isoformat()
        
        if self.config.verbose:
            print(f"[MemoryForgetter] 降权: {memory.id}")
            print(f"  新衰减分数: {memory.decay_score:.3f}")
        
        return memory
    
    def archive(self, memory: MemoryNode) -> MemoryNode:
        """
        归档：将记忆移至冷存储
        
        Args:
            memory: 记忆节点
            
        Returns:
            归档后的记忆
        """
        memory.status = MemoryStatus.ARCHIVED.value
        memory.updated_at = datetime.now().isoformat()
        
        if self.config.verbose:
            print(f"[MemoryForgetter] 归档: {memory.id}")
            print(f"  内容: {memory.memory[:30]}...")
        
        return memory
    
    def delete(self, memory: MemoryNode) -> MemoryNode:
        """
        删除：标记为删除状态
        
        注意：通常不建议物理删除，而是标记状态
        
        Args:
            memory: 记忆节点
            
        Returns:
            删除后的记忆
        """
        memory.status = MemoryStatus.DELETED.value
        memory.updated_at = datetime.now().isoformat()
        
        if self.config.verbose:
            print(f"[MemoryForgetter] 删除: {memory.id}")
            print(f"  内容: {memory.memory[:30]}...")
        
        return memory
    
    def auto_cleanup(self, memories: List[MemoryNode]) -> Dict[str, List[MemoryNode]]:
        """
        自动清理：根据衰减分数执行分层遗忘
        
        Args:
            memories: 记忆列表
            
        Returns:
            分类后的记忆字典 {action: [memories]}
        """
        # 首先更新衰减分数
        self.update_decay_scores(memories)
        
        result = {
            "kept": [],      # 保留
            "downweighted": [],  # 降权
            "archived": [],  # 归档
            "deleted": []    # 删除
        }
        
        for memory in memories:
            if memory.status != MemoryStatus.ACTIVATED.value:
                continue
            
            score = memory.decay_score
            
            if score >= self.config.archive_threshold:
                # 分数足够高，保留
                result["kept"].append(memory)
            elif score >= self.config.delete_threshold:
                # 分数较低，归档
                self.archive(memory)
                result["archived"].append(memory)
            else:
                # 分数极低，可考虑删除（但通常只是归档）
                self.archive(memory)  # 保守策略：归档而非删除
                result["archived"].append(memory)
        
        if self.config.verbose:
            print(f"\n[MemoryForgetter] 自动清理完成:")
            print(f"  保留: {len(result['kept'])} 条")
            print(f"  归档: {len(result['archived'])} 条")
        
        return result
    
    def reinforce(self, memory: MemoryNode) -> MemoryNode:
        """
        强化：当记忆被访问时，增强其保留概率
        
        Args:
            memory: 记忆节点
            
        Returns:
            强化后的记忆
        """
        memory.access_count += 1
        memory.last_accessed = datetime.now().isoformat()
        memory.decay_score = min(memory.decay_score * 1.2, 1.0)  # 提升20%
        
        if self.config.verbose:
            print(f"[MemoryForgetter] 强化: {memory.id}")
            print(f"  访问次数: {memory.access_count}")
            print(f"  新衰减分数: {memory.decay_score:.3f}")
        
        return memory


# ============================================================
# 遗忘策略示例
# ============================================================

def example_forgetting_strategy():
    """
    遗忘策略示例：展示衰减计算和自动清理
    """
    print("\n" + "="*60)
    print("【遗忘策略示例】")
    print("="*60)
    
    config = MemoryUpdateConfig(verbose=True)
    forgetter = MemoryForgetter(config)
    
    # 创建不同时间和访问频率的记忆
    now = datetime.now()
    
    memories = [
        # 最近创建，高访问量 -> 应该保留
        MemoryNode(
            id="mem_recent_hot",
            key="热门记忆",
            memory="用户最近频繁查询的热门话题",
            tags=["热门"],
            type=MemoryType.FACT.value,
            confidence=0.9,
            created_at=(now - timedelta(days=1)).isoformat(),
            updated_at=(now - timedelta(hours=2)).isoformat(),
            user_id="user_001",
            session_id="session_001",
            access_count=15,
            last_accessed=(now - timedelta(hours=1)).isoformat()
        ),
        # 较旧，中等访问量 -> 可能降权
        MemoryNode(
            id="mem_old_medium",
            key="中等记忆",
            memory="一个月前的记忆，偶尔被访问",
            tags=["普通"],
            type=MemoryType.FACT.value,
            confidence=0.8,
            created_at=(now - timedelta(days=30)).isoformat(),
            updated_at=(now - timedelta(days=20)).isoformat(),
            user_id="user_001",
            session_id="session_002",
            access_count=3,
            last_accessed=(now - timedelta(days=10)).isoformat()
        ),
        # 很旧，低访问量 -> 应该归档
        MemoryNode(
            id="mem_very_old",
            key="陈旧记忆",
            memory="半年前的记忆，几乎没有被访问",
            tags=["陈旧"],
            type=MemoryType.FACT.value,
            confidence=0.7,
            created_at=(now - timedelta(days=180)).isoformat(),
            updated_at=(now - timedelta(days=180)).isoformat(),
            user_id="user_001",
            session_id="session_003",
            access_count=0,
            last_accessed=None
        )
    ]
    
    print("\n--- 衰减分数计算 ---")
    for mem in memories:
        decay = forgetter.calculate_decay(mem, now)
        print(f"记忆: {mem.key}")
        print(f"  访问次数: {mem.access_count}")
        print(f"  衰减分数: {decay:.4f}")
        print()
    
    print("\n--- 自动清理 ---")
    result = forgetter.auto_cleanup(memories)
    
    print("\n--- 记忆强化示例 ---")
    # 模拟访问一条记忆
    if result["kept"]:
        mem_to_reinforce = result["kept"][0]
        print(f"强化前 - 访问次数: {mem_to_reinforce.access_count}, 衰减分数: {mem_to_reinforce.decay_score:.3f}")
        forgetter.reinforce(mem_to_reinforce)
        print(f"强化后 - 访问次数: {mem_to_reinforce.access_count}, 衰减分数: {mem_to_reinforce.decay_score:.3f}")

# 运行示例
example_forgetting_strategy()



【遗忘策略示例】

--- 衰减分数计算 ---
记忆: 热门记忆
  访问次数: 15
  衰减分数: 1.0000

记忆: 中等记忆
  访问次数: 3
  衰减分数: 0.8779

记忆: 陈旧记忆
  访问次数: 0
  衰减分数: 0.1000


--- 自动清理 ---
[MemoryForgetter] 低衰减分数: mem_very_old
  内容: 半年前的记忆，几乎没有被访问...
  衰减分数: 0.100
[MemoryForgetter] 归档: mem_very_old
  内容: 半年前的记忆，几乎没有被访问...

[MemoryForgetter] 自动清理完成:
  保留: 2 条
  归档: 1 条

--- 记忆强化示例 ---
强化前 - 访问次数: 15, 衰减分数: 1.000
[MemoryForgetter] 强化: mem_recent_hot
  访问次数: 16
  新衰减分数: 1.000
强化后 - 访问次数: 16, 衰减分数: 1.000


In [9]:
# ============================================================
# 记忆更新管理器 - Memory Update Manager (整合模块)
# ============================================================

class MemoryUpdateManager:
    """
    记忆更新管理器：整合所有更新模块的主入口
    
    功能：
    1. 统一管理记忆的增删改查
    2. 自动检测冲突并消解
    3. 定期执行遗忘清理
    4. 维护版本历史
    """
    
    def __init__(self, config: MemoryUpdateConfig = None):
        self.config = config or MemoryUpdateConfig()
        
        # 记忆存储
        self.memory_store: Dict[str, MemoryNode] = {}
        
        # 子模块
        self.merger = MemoryMerger(config)
        self.overwriter = MemoryOverwriter(config)
        self.version_manager = MemoryVersionManager(config)
        self.conflict_resolver = ConflictResolver(config)
        self.forgetter = MemoryForgetter(config)
        
        # 检索器
        self.retriever = SimpleRetriever(self.memory_store)
    
    def add_memory(self, memory: MemoryNode, 
                   check_conflict: bool = True,
                   check_merge: bool = True) -> Dict:
        """
        添加新记忆（带冲突检测和合并检查）
        
        Args:
            memory: 新记忆
            check_conflict: 是否检查冲突
            check_merge: 是否检查可合并项
            
        Returns:
            操作结果
        """
        result = {
            "action": "add",
            "memory_id": memory.id,
            "conflicts": [],
            "merged": False
        }
        
        # 1. 检索相关记忆
        similar_memories = self.retriever.find_similar(memory, threshold=0.3)
        
        # 2. 冲突检测
        if check_conflict and similar_memories:
            existing = [m for m, _ in similar_memories]
            conflicts = self.conflict_resolver.detect_conflict(memory, existing)
            
            if conflicts:
                result["conflicts"] = conflicts
                # 自动消解第一个冲突
                resolution = self.conflict_resolver.auto_resolve(
                    conflicts[0], 
                    {**self.memory_store, memory.id: memory}
                )
                result["resolution"] = resolution
                
                # 根据解决方案执行操作
                if resolution.get("action") == UpdateAction.OVERWRITE:
                    deprecate_id = resolution.get("deprecate_id")
                    if deprecate_id and deprecate_id in self.memory_store:
                        self.memory_store[deprecate_id] = self.overwriter.deprecate(
                            self.memory_store[deprecate_id],
                            reason="冲突消解：被新记忆替代"
                        )
        
        # 3. 合并检查
        if check_merge and similar_memories:
            mergeable = []
            for existing_mem, similarity in similar_memories:
                if self.merger.should_merge(memory, existing_mem, similarity):
                    mergeable.append(existing_mem)
            
            if mergeable:
                # 执行合并
                all_to_merge = [memory] + mergeable
                merged_memory, original_ids = self.merger.merge_batch(all_to_merge)
                
                # 废弃原记忆
                for oid in original_ids:
                    if oid in self.memory_store:
                        self.memory_store[oid].status = MemoryStatus.DEPRECATED.value
                
                # 存储合并后的记忆
                self.memory_store[merged_memory.id] = merged_memory
                result["action"] = "merge"
                result["memory_id"] = merged_memory.id
                result["merged"] = True
                result["merged_from"] = original_ids
                
                return result
        
        # 4. 直接添加
        self.memory_store[memory.id] = memory
        
        if self.config.verbose:
            print(f"[MemoryUpdateManager] 添加记忆: {memory.id}")
        
        return result
    
    def update_memory(self, memory_id: str, new_content: str,
                      use_versioning: bool = False) -> Dict:
        """
        更新记忆内容
        
        Args:
            memory_id: 记忆ID
            new_content: 新内容
            use_versioning: 是否使用版本化（保留历史）
            
        Returns:
            操作结果
        """
        if memory_id not in self.memory_store:
            return {"error": f"记忆不存在: {memory_id}"}
        
        old_memory = self.memory_store[memory_id]
        
        if use_versioning:
            # 版本化更新
            archived, new_version = self.version_manager.create_version(
                old_memory, new_content, "内容更新"
            )
            self.memory_store[old_memory.id] = archived
            self.memory_store[new_version.id] = new_version
            
            return {
                "action": "version",
                "old_id": old_memory.id,
                "new_id": new_version.id,
                "version": new_version.version
            }
        else:
            # 覆盖更新
            updated = self.overwriter.overwrite(old_memory, new_content)
            self.memory_store[memory_id] = updated
            
            return {
                "action": "overwrite",
                "memory_id": memory_id,
                "version": updated.version
            }
    
    def access_memory(self, memory_id: str) -> Optional[MemoryNode]:
        """
        访问记忆（会触发强化）
        
        Args:
            memory_id: 记忆ID
            
        Returns:
            记忆节点
        """
        if memory_id not in self.memory_store:
            return None
        
        memory = self.memory_store[memory_id]
        
        if memory.status == MemoryStatus.ACTIVATED.value:
            # 强化记忆
            self.forgetter.reinforce(memory)
        
        return memory
    
    def run_cleanup(self) -> Dict:
        """
        执行清理任务
        
        Returns:
            清理结果
        """
        active_memories = [
            m for m in self.memory_store.values() 
            if m.status == MemoryStatus.ACTIVATED.value
        ]
        
        result = self.forgetter.auto_cleanup(active_memories)
        
        return {
            "total_processed": len(active_memories),
            "kept": len(result["kept"]),
            "archived": len(result["archived"])
        }
    
    def get_stats(self) -> Dict:
        """获取记忆库统计信息"""
        stats = {
            "total": len(self.memory_store),
            "activated": 0,
            "archived": 0,
            "deprecated": 0,
            "deleted": 0
        }
        
        for memory in self.memory_store.values():
            if memory.status == MemoryStatus.ACTIVATED.value:
                stats["activated"] += 1
            elif memory.status == MemoryStatus.ARCHIVED.value:
                stats["archived"] += 1
            elif memory.status == MemoryStatus.DEPRECATED.value:
                stats["deprecated"] += 1
            elif memory.status == MemoryStatus.DELETED.value:
                stats["deleted"] += 1
        
        return stats


# ============================================================
# 完整流程示例
# ============================================================

def example_full_workflow():
    """
    完整工作流示例：展示记忆更新的完整生命周期
    """
    print("\n" + "="*60)
    print("【完整工作流示例】")
    print("="*60)
    
    # 初始化管理器
    config = MemoryUpdateConfig(verbose=True)
    manager = MemoryUpdateManager(config)
    
    # 1. 添加初始记忆
    print("\n--- 步骤1：添加初始记忆 ---")
    initial_memories = [
        MemoryNode(
            id="pref_001",
            key="饮品偏好",
            memory="用户喜欢喝咖啡，尤其是美式咖啡",
            tags=["偏好", "饮品", "咖啡"],
            type=MemoryType.PREFERENCE.value,
            confidence=0.9,
            created_at="2025-06-01T10:00:00",
            updated_at="2025-06-01T10:00:00",
            user_id="user_001",
            session_id="session_001"
        ),
        MemoryNode(
            id="fact_001",
            key="工作信息",
            memory="用户是一名软件工程师，在科技公司工作",
            tags=["事实", "工作"],
            type=MemoryType.FACT.value,
            confidence=0.95,
            created_at="2025-06-01T10:00:00",
            updated_at="2025-06-01T10:00:00",
            user_id="user_001",
            session_id="session_001"
        )
    ]
    
    for mem in initial_memories:
        result = manager.add_memory(mem, check_conflict=False, check_merge=False)
        print(f"添加结果: {result}")
    
    print(f"\n当前统计: {manager.get_stats()}")
    
    # 2. 添加可能冲突的记忆
    print("\n--- 步骤2：添加冲突记忆 ---")
    conflicting_memory = MemoryNode(
        id="pref_002",
        key="饮品偏好",
        memory="用户说不喜欢咖啡了，改喝茶",
        tags=["偏好", "饮品", "茶"],
        type=MemoryType.PREFERENCE.value,
        confidence=0.95,
        created_at="2026-01-10T10:00:00",
        updated_at="2026-01-10T10:00:00",
        user_id="user_001",
        session_id="session_010",
        source_credibility=1.0  # 用户明确陈述
    )
    
    result = manager.add_memory(conflicting_memory)
    print(f"添加结果: {result}")
    
    # 3. 更新记忆（使用版本化）
    print("\n--- 步骤3：版本化更新 ---")
    result = manager.update_memory(
        "fact_001",
        "用户是一名高级软件工程师，最近晋升为技术主管",
        use_versioning=True
    )
    print(f"更新结果: {result}")
    
    # 4. 访问记忆（触发强化）
    print("\n--- 步骤4：访问记忆 ---")
    accessed = manager.access_memory("pref_002")
    if accessed:
        print(f"访问记忆: {accessed.key}")
        print(f"  访问次数: {accessed.access_count}")
        print(f"  衰减分数: {accessed.decay_score:.3f}")
    
    # 5. 执行清理
    print("\n--- 步骤5：执行清理 ---")
    cleanup_result = manager.run_cleanup()
    print(f"清理结果: {cleanup_result}")
    
    # 6. 最终统计
    print("\n--- 最终统计 ---")
    print(f"记忆库状态: {manager.get_stats()}")
    
    print("\n所有记忆:")
    for mid, mem in manager.memory_store.items():
        print(f"  [{mem.status}] {mid}: {mem.memory[:40]}...")

# 运行完整示例
example_full_workflow()



【完整工作流示例】

--- 步骤1：添加初始记忆 ---
[MemoryUpdateManager] 添加记忆: pref_001
添加结果: {'action': 'add', 'memory_id': 'pref_001', 'conflicts': [], 'merged': False}
[MemoryUpdateManager] 添加记忆: fact_001
添加结果: {'action': 'add', 'memory_id': 'fact_001', 'conflicts': [], 'merged': False}

当前统计: {'total': 2, 'activated': 2, 'archived': 0, 'deprecated': 0, 'deleted': 0}

--- 步骤2：添加冲突记忆 ---
[LLMService] API调用失败: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-api*****here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
[LLMService] JSON解析失败: Expecting value: line 1 column 1 (char 0)
[MemoryUpdateManager] 添加记忆: pref_002
添加结果: {'action': 'add', 'memory_id': 'pref_002', 'conflicts': [], 'merged': False}

--- 步骤3：版本化更新 ---
[MemoryVersionManager] 创建新版本
  旧版本ID: fact_001 (v1) -> 已归档
  新版本ID: 7b149243-1144-45a5-a356-25952e941975 (v2)
  变更描述: 内容更新
更新结果: {'action': 'version', 'old_id':

In [10]:
# ============================================================
# 总结 / Summary
# ============================================================
"""
本章实现了记忆更新的完整模块，所有智能判断均使用 OpenAI API (gpt-4.1-mini) 实现：

1. LLMService - 核心LLM服务
   - detect_conflict(): 使用LLM检测记忆冲突
   - merge_memories(): 使用LLM智能合并多条记忆
   - should_merge(): 使用LLM判断是否应该合并
   - resolve_conflict(): 使用LLM解决记忆冲突
   - determine_update_action(): 使用LLM决定更新操作

2. 记忆覆盖 (MemoryOverwriter)
   - 适用于用户明确修改、配置替换、纠错等场景
   - 直接用新内容替代旧内容，可选择保留或废弃旧记忆

3. 记忆合并 (MemoryMerger) 【核心示例】
   - 使用LLM智能判断是否应该合并
   - 使用LLM生成合并后的综合记忆
   - 合并策略：内容合并、标签并集、置信度取最高

4. 记忆版本化 (MemoryVersionManager)
   - 适用于需要审计追溯的场景
   - 保留完整版本历史，支持回滚
   - 维护版本链和变更日志

5. 冲突消解 (ConflictResolver)
   - 使用LLM检测显式事实冲突和隐式偏好冲突
   - 使用LLM智能选择解决策略
   - 支持：保留A/保留B/合并/共存降权

6. 遗忘策略 (MemoryForgetter)
   - 类比艾宾浩斯遗忘曲线的衰减机制
   - 四种遗忘动作：降权、归档、压缩、删除
   - 访问强化机制

7. 整合管理器 (MemoryUpdateManager)
   - 统一管理记忆的增删改查
   - 自动检测冲突并使用LLM消解
   - 定期执行遗忘清理

核心要点：
- 所有智能判断（冲突检测、合并决策、冲突消解）均由LLM完成
- 记忆更新不是可选功能，而是维持系统长期可靠性的必要机制
- 没有更新能力的记忆系统会累积噪声、过时信息和冲突
- 遗忘是维持系统健康的必要能力，而非缺陷
- 版本化提供了可追溯性和回滚能力
"""

print("="*60)
print("记忆更新模块实现完成！（使用OpenAI API）")
print("="*60)
print(f"""
配置信息：
├── API Base URL: {OPENAI_BASE_URL}
├── Model: {MODEL_NAME}
└── LLM Service: {'已初始化' if llm_service else '未初始化'}

模块清单：
├── LLMService          - LLM智能服务（OpenAI API）
├── SimpleRetriever     - 简单检索器
├── MemoryNode          - 记忆节点数据结构
├── MemoryMerger        - 记忆合并器 【书本示例】
├── MemoryOverwriter    - 记忆覆盖器
├── MemoryVersionManager- 版本管理器
├── ConflictResolver    - 冲突消解器
├── MemoryForgetter     - 遗忘策略器
└── MemoryUpdateManager - 整合管理器

示例函数：
├── example_memory_merge()           - 记忆合并示例
├── example_memory_overwrite_and_version() - 覆盖与版本化示例
├── example_conflict_resolution()    - 冲突消解示例
├── example_forgetting_strategy()    - 遗忘策略示例
└── example_full_workflow()          - 完整工作流示例
""")


记忆更新模块实现完成！（使用OpenAI API）

配置信息：
├── API Base URL: https://api.openai.com/v1
├── Model: gpt-4.1-mini
└── LLM Service: 已初始化

模块清单：
├── LLMService          - LLM智能服务（OpenAI API）
├── SimpleRetriever     - 简单检索器
├── MemoryNode          - 记忆节点数据结构
├── MemoryMerger        - 记忆合并器 【书本示例】
├── MemoryOverwriter    - 记忆覆盖器
├── MemoryVersionManager- 版本管理器
├── ConflictResolver    - 冲突消解器
├── MemoryForgetter     - 遗忘策略器
└── MemoryUpdateManager - 整合管理器

示例函数：
├── example_memory_merge()           - 记忆合并示例
├── example_memory_overwrite_and_version() - 覆盖与版本化示例
├── example_conflict_resolution()    - 冲突消解示例
├── example_forgetting_strategy()    - 遗忘策略示例
└── example_full_workflow()          - 完整工作流示例

